# (연구&재인) MBTI(정리) – 실험셋업 시각화

신록예찬  
2023-12-20

# 1. Imports

In [67]:
import pandas as pd
import time
import pickle
import warnings
warnings.filterwarnings('ignore')

In [68]:
import plotly.express as px
import plotly.io as pio
from plotly.subplots import make_subplots
pd.options.plotting.backend = "plotly"
pio.templates.default = "plotly_white"

# 2. Data

In [69]:
!ls '정리된자료(csv)'

df_gpt.csv           tidydata_실험3시나리오0.csv
df_kaggle.csv            tidydata_실험3시나리오1.csv
tidydata_실험1시나리오1.csv  tidydata_실험3시나리오2.csv
tidydata_실험1시나리오2.csv  tidydata_실험3시나리오3.csv
tidydata_실험2시나리오0.csv  tidydata_실험3시나리오4.csv
tidydata_실험2시나리오1.csv  tidydata_실험3시나리오5.csv
tidydata_실험2시나리오2.csv  tidydata_실험3시나리오6.csv
tidydata_실험2시나리오3.csv  tidydata_실험3시나리오7.csv
tidydata_실험2시나리오4.csv

In [70]:
df_kaggle = pd.read_csv("정리된자료(csv)/df_kaggle.csv")
df_gpt = pd.read_csv("정리된자료(csv)/df_gpt.csv")

# 3. 실험1의 계획 시각화

`-` kaggle 데이터만

In [71]:
fig = df_kaggle.type.value_counts().reset_index()\
.plot.bar(
    x='type',y='count',
    text='count',
    title="",
    width=900,
    height=400
)
fig.layout['xaxis']['title']['text'] = ''
fig.layout['yaxis']['title']['text'] = ''
with open('Figures/원래자료.pkl','wb') as f:
    pickle.dump(fig,f)

`-` kaggle 데이터 + count\<100 하이라이팅

In [72]:
fig = df_kaggle.type.value_counts().reset_index().eval('gen = count<100').rename({'gen':'Count<100'},axis=1)\
.plot.bar(
    x='type',y='count',
    text='count',
    color='Count<100',
    opacity=1.0,
    width=900,
    height=400    
)
fig['data'][1]['marker']['color']='#636efa'
fig['data'][0]['marker']['opacity']=0.2
fig['data'][0]['showlegend'] = False
fig['data'][1]['showlegend'] = False
fig.layout['xaxis']['title']['text'] = ''
fig.layout['yaxis']['title']['text'] = ''
with open('Figures/실험셋업시각화_실험1실험계획.pkl','wb') as f:
    pickle.dump(fig,f)

# 4. 실험1의 셋팅들 시각화

In [73]:
tidydata = pd.read_csv("정리된자료(csv)/tidydata_실험1시나리오1.csv")
fig = px.bar(
    tidydata,
    x='type',
    y='count',
    text='count',
    pattern_shape='DataType',
    barmode='group',
    width=900,
    height=400        
)
fig.layout['xaxis']['title']['text'] = ''
fig.layout['yaxis']['title']['text'] = ''
with open('Figures/실험셋업시각화_실험1시나리오1.pkl','wb') as f:
    pickle.dump(fig,f)

In [74]:
tidydata = pd.read_csv("정리된자료(csv)/tidydata_실험1시나리오2.csv")
fig = px.bar(
    tidydata,
    x='type',
    y='count',
    color=tidydata['col'],
    text='count',
    pattern_shape='DataType',
    barmode='group',
    hover_data='Source',
    width=900,
    height=400    
)
fig['data'][0]['marker']['color'] = ['#636efa']*16 + ['#EF553B']*4 
fig['data'][1]['marker']['color'] = ['#636efa']*20
fig['data'][0]['hovertemplate']='Type=%{x}<br>Count=%{text}<br>DataType=Train<br>Source=%{customdata[0]}<br><extra></extra>'
fig['data'][1]['hovertemplate']='Type=%{x}<br>Count=%{text}<br>DataType=Train<br>Source=%{customdata[0]}<br><extra></extra>'
fig.layout['xaxis']['title']['text'] = ''
fig.layout['yaxis']['title']['text'] = ''
with open('Figures/실험셋업시각화_실험1시나리오2.pkl','wb') as f:
    pickle.dump(fig,f)

# 5. 실험2의 셋팅들 시각화

In [75]:
df0 = pd.read_csv("정리된자료(csv)/tidydata_실험2시나리오0.csv").query('Setting != "Real+Synthetic"').assign(시나리오 = '0')[:2]
df1 = pd.read_csv("정리된자료(csv)/tidydata_실험2시나리오1.csv").query('Setting != "Real+Synthetic"').assign(시나리오 = '1')
df2 = pd.read_csv("정리된자료(csv)/tidydata_실험2시나리오2.csv").query('Setting != "Real+Synthetic"').assign(시나리오 = '2')
df3 = pd.read_csv("정리된자료(csv)/tidydata_실험2시나리오3.csv").query('Setting != "Real+Synthetic"').assign(시나리오 = '3')
tidydata = pd.concat([df0,df1,df2,df3]).reset_index(drop=True).assign(text = lambda df: [f'{c} ({s})' for s,c in zip(df.Source,df['count'].astype(str))])
fig = px.bar(
    tidydata,
    y='type',
    x='count',
    color='col',
    pattern_shape='DataType',
    category_orders={'DataType':{'Train','Test'}},
    barmode='group',
    hover_data='Source',
    text='text',
    facet_col='Setting',
    facet_row='시나리오',
    width=900,
    height=400
)
for i in range(len(fig['data'])):
    fig['data'][i]['marker']['color'] = list(pd.Series(fig['data'][i]['marker']['color']).map({0:'#636efa',1:'#EF553B'}))
    fig['data'][i]['hovertemplate']='Type=%{x}<br>Count=%{text}<br>DataType=Train<br>Source=%{customdata[0]}<br><extra></extra>'
    if fig['data'][i]['marker']['pattern']['shape'] == '/':
        fig['data'][i]['marker']['pattern']['shape'] = ''
    else:
        fig['data'][i]['marker']['pattern']['shape'] = '/'
fig.layout['legend']['title']['text']=''
fig.layout['annotations'][0]['text'] = ""
fig.layout['annotations'][1]['text'] = ""
for ann in fig['layout']['annotations']:
    ann['text'] =''
fig.layout['xaxis']['title']['text']=''
fig.layout['xaxis2']['title']['text']=''
fig.layout['yaxis']['title']['text']=''
fig.layout['yaxis3']['title']['text']=''
fig.layout['yaxis5']['title']['text']=''
fig.layout['yaxis7']['title']['text']=''
fig.layout['xaxis']['title']['text'] = ''
fig.layout['yaxis']['title']['text'] = ''
with open('Figures/실험셋업시각화_실험2시나리오0-3.pkl','wb') as f:
    pickle.dump(fig,f)

# 5. 실험3의 셋팅들 시각화

In [76]:
df0 = pd.read_csv("정리된자료(csv)/tidydata_실험3시나리오0.csv").query('Setting != "Real+Synthetic"').assign(시나리오 = '0')
df1 = pd.read_csv("정리된자료(csv)/tidydata_실험3시나리오1.csv").query('Setting != "Real+Synthetic"').assign(시나리오 = '1')
df2 = pd.read_csv("정리된자료(csv)/tidydata_실험3시나리오2.csv").query('Setting != "Real+Synthetic"').assign(시나리오 = '2')
df3 = pd.read_csv("정리된자료(csv)/tidydata_실험3시나리오3.csv").query('Setting != "Real+Synthetic"').assign(시나리오 = '3')
df4 = pd.read_csv("정리된자료(csv)/tidydata_실험3시나리오4.csv").query('Setting != "Real+Synthetic"').assign(시나리오 = '4')
df5 = pd.read_csv("정리된자료(csv)/tidydata_실험3시나리오5.csv").query('Setting != "Real+Synthetic"').assign(시나리오 = '5')
df6 = pd.read_csv("정리된자료(csv)/tidydata_실험3시나리오6.csv").query('Setting != "Real+Synthetic"').assign(시나리오 = '6')
df7 = pd.read_csv("정리된자료(csv)/tidydata_실험3시나리오7.csv").query('Setting != "Real+Synthetic"').assign(시나리오 = '7')
tidydata = pd.concat([df0,df1,df2,df3,df4,df5,df6,df7]).reset_index(drop=True).assign(text = lambda df: [f'{c} ({s})' for s,c in zip(df.Source,df['count'].astype(str))])

In [77]:
# fig = px.bar(
#     tidydata.query("시나리오 =='0'"),
#     x='type',
#     y='count',
#     color='col',
#     pattern_shape='DataType',
#     barmode='group',
#     hover_data='Source',
#     facet_col='Setting',
#     width=900,
#     height=400    
# )
# for geom in fig['data']:
#     geom['marker']['color'] = list(pd.Series(geom['marker']['color']).map({0:'#636efa',1:'#636efa'}))       
# fig.layout['legend']['title']['text']=''
# fig.layout['annotations'][0]['text'] = ""
# for ann in fig['layout']['annotations']:
#     ann['text'] =''
# with open('Figures/실험셋업시각화_실험3시나리오0.pkl','wb') as f:
#     pickle.dump(fig,f)
# fig.write_image('Figures/실험셋업시각화_실험3시나리오0.svg')
# fig

In [78]:
for i in '1234567': 
    fig = px.bar(
        tidydata.query(f"시나리오 =='{i}'"),
        x='type',
        y='count',
        color='col',
        pattern_shape='DataType',
        barmode='group',
        hover_data='Source',
        facet_col='Setting',
        width=900,
        height=400
    )
    for geom in fig['data']:
        geom['marker']['color'] = list(pd.Series(geom['marker']['color']).map({0:'#636efa',1:'#EF553B'}))       
    fig.layout['legend']['title']['text']=''
    for ann in fig['layout']['annotations']:
        ann['text'] =''
    with open(f'Figures/실험셋업시각화_실험3시나리오{i}.pkl','wb') as f:
        pickle.dump(fig,f)